# Seaborn — Beautiful Statistical Visualization

## What is Seaborn?

Seaborn is a Python visualization library built **on top of Matplotlib** that makes **statistical plots beautiful and easy**. While Matplotlib gives you raw control, Seaborn gives you smart defaults and powerful statistical features in much less code.

**Real-world analogy**: If Matplotlib is a raw lumber yard where you build everything from scratch, Seaborn is IKEA — pre-designed, good-looking furniture that you assemble quickly. When you need something custom, you still drop down to Matplotlib (the lumber yard).

## Why Seaborn?

| Without Seaborn | With Seaborn |
|----------------|-------------|
| 15 lines to make a violin plot | 1 line: `sns.violinplot(...)` |
| Manual color palettes | Built-in beautiful palettes |
| Separate loop for group colors | `hue='category'` parameter |
| No confidence intervals | Automatic CIs on line/bar plots |
| Static theme | `sns.set_theme()` for instant polish |

## Prerequisites
- Python basics
- Pandas DataFrames (Seaborn works natively with DataFrames)
- Matplotlib basics (for customization)

## Table of Contents
1. Installation & Setup
2. Seaborn's Mental Model: `data`, `x`, `y`, `hue`
3. Distribution Plots (histplot, kdeplot, ecdfplot, rugplot)
4. Categorical Plots (boxplot, violinplot, stripplot, barplot, countplot)
5. Relational Plots (scatterplot, lineplot)
6. Regression Plots (regplot, lmplot)
7. Matrix Plots (heatmap, clustermap)
8. Figure-Level vs Axes-Level Functions
9. Multi-Plot Grids (FacetGrid, PairGrid, pairplot)
10. Themes & Color Palettes
11. Combining Seaborn and Matplotlib
12. Common Pitfalls
13. Mini Project: Exploratory Data Analysis (EDA) Pipeline
14. Interview Q&A
15. Resources

---

**Official Docs**: https://seaborn.pydata.org/  
**API Reference**: https://seaborn.pydata.org/api.html  
**Gallery**: https://seaborn.pydata.org/examples/  
**YouTube (Keith Galli)**: https://www.youtube.com/watch?v=6GUZXDef2U0  
**Tutorial Blog (Michael Waskom — Seaborn creator)**: https://seaborn.pydata.org/tutorial.html

## 1. Installation & Setup

```bash
pip install seaborn matplotlib pandas numpy
```

In [ ]:
seaborntry:
    import seaborn as sns
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    print(f"Seaborn {sns.__version__} ready")
except ImportError:
    raise SystemExit("Run: pip install seaborn matplotlib pandas numpy")


---
## 2. Seaborn's Mental Model: `data`, `x`, `y`, `hue`

Almost every Seaborn function takes the same core arguments:

```python
sns.plot_type(data=df, x='column_name', y='column_name', hue='category_column')
```

| Parameter | Meaning |
|-----------|---------|
| `data` | A Pandas DataFrame |
| `x` | Column name for the x-axis |
| `y` | Column name for the y-axis |
| `hue` | Column name for color grouping (adds a legend) |
| `size` | Column name for point size |
| `style` | Column name for marker style |
| `col` / `row` | Create a grid: one subplot per value in this column |

**Example**: To visualize tip amounts by day, colored by whether it's lunch or dinner:
```python
sns.boxplot(data=tips, x='day', y='tip', hue='time')
```
Three words do the work of 20 lines of Matplotlib code.

In [ ]:
# Load the classic 'tips' dataset (restaurant bill data)
tips = sns.load_dataset('tips')
print("Tips dataset shape:", tips.shape)
print("\nFirst 5 rows:")
print(tips.head())
print("\nColumn types:")
print(tips.dtypes)
print("\nBasic stats:")
print(tips.describe())

---
## 3. Distribution Plots

These plots answer: **"What does my data look like? What's the shape of its distribution?"**

| Function | Shows | Use When |
|----------|-------|----------|
| `histplot` | Bars = count/density per bin | Always start here |
| `kdeplot` | Smooth estimated density curve | Overlaying groups |
| `ecdfplot` | Empirical CDF (cumulative) | Comparing percentiles |
| `rugplot` | Rug marks at each data point | On top of other plots |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── 1. Histogram with KDE overlay ─────────────────────────────────────────────
sns.histplot(data=tips, x='total_bill', bins=25, kde=True, 
             color='steelblue', ax=axes[0, 0])
axes[0, 0].set_title('Histogram + KDE: Total Bill Distribution')
axes[0, 0].axvline(tips['total_bill'].mean(), color='red', linestyle='--', 
                    label=f'Mean: ${tips["total_bill"].mean():.2f}')
axes[0, 0].legend()

# ── 2. Histogram with hue (grouped by category) ────────────────────────────────
sns.histplot(data=tips, x='total_bill', hue='time', bins=20, 
             multiple='stack', ax=axes[0, 1])  # 'stack', 'dodge', 'fill', 'layer'
axes[0, 1].set_title('Stacked Histogram: Bill by Meal Time')

# ── 3. KDE plot — smooth distributions ────────────────────────────────────────
sns.kdeplot(data=tips, x='total_bill', hue='day', 
             fill=True, alpha=0.3, ax=axes[1, 0])  # fill=True shades under curve
axes[1, 0].set_title('KDE: Bill Distribution by Day')

# ── 4. ECDF: Empirical Cumulative Distribution ─────────────────────────────────
sns.ecdfplot(data=tips, x='total_bill', hue='sex', ax=axes[1, 1])
axes[1, 1].axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='50th percentile')
axes[1, 1].set_title('ECDF: "What % of bills are below $X?"')
axes[1, 1].set_xlabel('Total Bill ($)')
axes[1, 1].set_ylabel('Cumulative Proportion')

plt.suptitle('Distribution Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Categorical Plots

These answer: **"How does a numeric variable differ across categories?"**

| Function | Shows | Use When |
|----------|-------|----------|
| `boxplot` | Median, quartiles, outliers | Comparing spread across groups |
| `violinplot` | Full distribution shape | When you want more than a box |
| `stripplot` | Individual data points | Showing raw data (small n) |
| `swarmplot` | Points that don't overlap | Small-medium datasets |
| `barplot` | Mean + confidence interval | Statistical comparison of means |
| `countplot` | Count of each category | Like a bar chart for counts |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# ── Box Plot ───────────────────────────────────────────────────────────────────
sns.boxplot(data=tips, x='day', y='total_bill', hue='sex', ax=axes[0, 0],
             palette='Set2', order=['Thur', 'Fri', 'Sat', 'Sun'])
axes[0, 0].set_title('Box Plot: Bill by Day & Gender')
axes[0, 0].set_xlabel('')

# ── Violin Plot ────────────────────────────────────────────────────────────────
sns.violinplot(data=tips, x='day', y='total_bill', hue='time', ax=axes[0, 1],
                split=True,  # split=True: left/right halves for each hue
                palette='muted', inner='quartile',  # show quartiles inside
                order=['Thur', 'Fri', 'Sat', 'Sun'])
axes[0, 1].set_title('Violin Plot: Split by Meal Time')
axes[0, 1].set_xlabel('')

# ── Strip Plot (individual points) ────────────────────────────────────────────
sns.stripplot(data=tips, x='day', y='tip', hue='sex', ax=axes[0, 2],
               jitter=True, alpha=0.7, dodge=True,
               order=['Thur', 'Fri', 'Sat', 'Sun'])
axes[0, 2].set_title('Strip Plot: Individual Tips')
axes[0, 2].set_xlabel('')

# ── Bar Plot (mean + 95% CI) ───────────────────────────────────────────────────
# errorbar='ci' (default) shows 95% bootstrap confidence interval
sns.barplot(data=tips, x='day', y='total_bill', hue='time', ax=axes[1, 0],
             estimator='mean', errorbar='ci',
             order=['Thur', 'Fri', 'Sat', 'Sun'])
axes[1, 0].set_title('Bar Plot: Mean Bill + 95% CI')
axes[1, 0].set_xlabel('')

# ── Count Plot ─────────────────────────────────────────────────────────────────
sns.countplot(data=tips, x='day', hue='sex', ax=axes[1, 1],
               order=['Thur', 'Fri', 'Sat', 'Sun'], palette='pastel')
axes[1, 1].set_title('Count Plot: Visits by Day & Gender')
axes[1, 1].set_xlabel('')

# ── Box + Strip combined (very common in data science) ────────────────────────
sns.boxplot(data=tips, x='smoker', y='tip', ax=axes[1, 2], 
             color='lightblue', width=0.5)
sns.stripplot(data=tips, x='smoker', y='tip', ax=axes[1, 2],
               jitter=True, alpha=0.5, color='darkblue', size=4)
axes[1, 2].set_title('Box + Strip: Tips by Smoker Status')

plt.suptitle('Categorical Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Relational Plots

These answer: **"What is the relationship between two numeric variables?"**

- `scatterplot`: each point = one observation
- `lineplot`: connects points in x-order; great for time series; automatically adds confidence intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Scatter Plot with hue, size, style ────────────────────────────────────────
sns.scatterplot(data=tips, x='total_bill', y='tip', 
                 hue='time',    # color = lunch/dinner
                 size='size',   # marker size = party size
                 style='sex',   # shape = male/female
                 alpha=0.7, ax=axes[0], palette='deep')
axes[0].set_title('Scatter: Bill vs Tip (hue=time, size=party, style=sex)')
axes[0].set_xlabel('Total Bill ($)')
axes[0].set_ylabel('Tip ($)')

# ── Line Plot: Time Series with Confidence Interval ──────────────────────────
# Generate time series data: weekly sales for two products
np.random.seed(42)
weeks = np.tile(np.arange(1, 53), 2)  # 52 weeks, 2 products
product = ['Product A'] * 52 + ['Product B'] * 52
# Simulate multiple observations per week (for CI)
# Seaborn's lineplot calculates CI by default across multiple observations at same x
all_weeks, all_sales, all_prod = [], [], []
for week in range(1, 53):
    for _ in range(5):  # 5 stores per week
        base_a = 100 + week * 2 + np.random.normal(0, 15)
        base_b = 80  + week * 1.5 + np.random.normal(0, 20)
        all_weeks += [week, week]
        all_sales += [base_a, base_b]
        all_prod  += ['Product A', 'Product B']

sales_df = pd.DataFrame({'Week': all_weeks, 'Sales': all_sales, 'Product': all_prod})
sns.lineplot(data=sales_df, x='Week', y='Sales', hue='Product', 
              ax=axes[1], palette=['steelblue', 'coral'], linewidth=2)
axes[1].set_title('Line Plot: Weekly Sales (shaded = 95% CI)')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Sales (units)')

plt.tight_layout()
plt.show()

print("Note: The shaded band in lineplot = 95% confidence interval")
print("Seaborn calculates this automatically when there are multiple y values per x!")

---
## 6. Regression Plots

Seaborn can fit and visualize regression lines directly in the plot:

- `regplot(x, y)`: scatter + linear regression line + CI
- `lmplot(x, y, data, hue/col/row)`: `regplot` with FacetGrid support (can split by category)
- `residplot(x, y)`: residuals of the regression (check if linear model fits)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── 1. Basic regression plot ──────────────────────────────────────────────────
sns.regplot(data=tips, x='total_bill', y='tip', ax=axes[0],
             scatter_kws={'alpha': 0.5, 's': 30}, line_kws={'color': 'red', 'linewidth': 2})
axes[0].set_title('regplot: Linear Regression\nBill → Tip')

# ── 2. Polynomial regression ──────────────────────────────────────────────────
np.random.seed(42)
x = np.linspace(0, 10, 80)
y = 2*x - 0.1*x**2 + np.random.normal(0, 2, 80)
poly_df = pd.DataFrame({'x': x, 'y': y})
sns.regplot(data=poly_df, x='x', y='y', ax=axes[1], order=2,  # order=2 → quadratic
             scatter_kws={'alpha': 0.4}, line_kws={'color': 'purple'})
axes[1].set_title('regplot: Polynomial Regression (order=2)')

# ── 3. Residuals plot ─────────────────────────────────────────────────────────
sns.residplot(data=tips, x='total_bill', y='tip', ax=axes[2],
               lowess=True, scatter_kws={'alpha': 0.5})
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_title('residplot: Residuals\n(good if randomly scattered around 0)')

plt.tight_layout()
plt.show()

# ── lmplot: Regression by group (creates its own figure) ─────────────────────
g = sns.lmplot(data=tips, x='total_bill', y='tip', hue='sex', 
                col='time',   # Separate column for Lunch vs Dinner
                palette='Set1', height=4, aspect=1.1,
                scatter_kws={'alpha': 0.4})
g.fig.suptitle('lmplot: Regression by Gender, Split by Meal Time', y=1.02)
plt.show()

---
## 7. Matrix Plots

Matrix plots visualize 2D data stored in matrices:

- **`heatmap`**: Color-coded matrix — perfect for correlation matrices and pivot tables
- **`clustermap`**: Heatmap + hierarchical clustering (groups similar rows/columns)

In [ ]:
# ── Generate a more interesting dataset ───────────────────────────────────────
np.random.seed(42)
n = 200
study_data = pd.DataFrame({
    'Math':     np.random.normal(75, 12, n),
    'Physics':  None,
    'Chemistry':None,
    'Biology':  np.random.normal(72, 10, n),
    'History':  np.random.normal(68, 11, n),
    'English':  np.random.normal(74, 9, n),
})
# Physics/Chemistry strongly correlated with Math
study_data['Physics']   = 0.8 * study_data['Math']   + np.random.normal(0, 5, n) + 10
study_data['Chemistry'] = 0.7 * study_data['Physics'] + np.random.normal(0, 6, n) + 5

corr_matrix = study_data.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Correlation Heatmap ────────────────────────────────────────────────────────
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show only lower triangle
sns.heatmap(corr_matrix, ax=axes[0],
             mask=mask,                   # Hide upper triangle (it's redundant)
             annot=True,                  # Print correlation values in cells
             fmt='.2f',                   # Format: 2 decimal places
             cmap='RdYlGn',              # Red=negative, Yellow=zero, Green=positive
             vmin=-1, vmax=1,             # Fix scale from -1 to +1
             linewidths=0.5,              # Lines between cells
             square=True,                 # Square cells
             cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation Heatmap: Student Grades', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# ── Monthly sales pivot heatmap ────────────────────────────────────────────────
# Create a product × month sales matrix
products = ['Laptop', 'Phone', 'Tablet', 'Watch', 'TV']
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
base_sales = np.array([30, 45, 20, 15, 25])
seasonal   = np.tile(np.array([0.9, 0.85, 1.0, 0.95, 1.05, 1.0, 1.0, 1.1, 0.9, 1.0, 1.2, 1.4]), (5, 1))
sales_matrix = (base_sales[:, None] * seasonal + np.random.normal(0, 2, (5, 12))).clip(0)
sales_df_pivot = pd.DataFrame(sales_matrix, index=products, columns=month_names)

sns.heatmap(sales_df_pivot, ax=axes[1], 
             annot=True, fmt='.0f', cmap='YlOrRd',
             linewidths=0.5, cbar_kws={'label': 'Units Sold'})
axes[1].set_title('Sales Heatmap: Product × Month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Product')

plt.tight_layout()
plt.show()

---
## 8. Figure-Level vs Axes-Level Functions

This is a critical Seaborn concept:

| Type | Examples | Returns | Works with `ax=` |
|------|----------|---------|------------------|
| **Axes-level** | `histplot`, `boxplot`, `scatterplot`, `heatmap` | `Axes` object | Yes |
| **Figure-level** | `displot`, `catplot`, `relplot`, `lmplot` | `FacetGrid` object | No (use `col`/`row`) |

**Rule of thumb**:
- Use **axes-level** functions when you want to put the plot in an existing subplot layout
- Use **figure-level** functions when you want automatic multi-panel plots with `col`/`row` splitting

In [ ]:
# ── Axes-level: goes into an existing subplot ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=tips, x='total_bill', kde=True, ax=axes[0])  # axes[0] specified!
sns.boxplot(data=tips, x='day', y='tip', ax=axes[1])
axes[0].set_title('Axes-level: inside plt.subplots()')
axes[1].set_title('Axes-level: in a different subplot')
plt.tight_layout()
plt.show()

# ── Figure-level: creates its own figure and axes ──────────────────────────────
# catplot can do boxplot/violin/strip/bar — controlled by `kind` parameter
g = sns.catplot(data=tips, x='day', y='tip', hue='sex', 
                 col='time',         # Creates 2 panels: Lunch | Dinner
                 kind='box',         # kind: 'box', 'violin', 'strip', 'bar', 'point'
                 height=4, aspect=1.0,
                 order=['Thur', 'Fri', 'Sat', 'Sun'])
g.set_titles(col_template='Meal Time: {col_name}')  # Custom panel titles
g.fig.suptitle('Figure-level: catplot (kind="box") — auto multi-panel', y=1.05)
plt.show()

print("Key difference: figure-level returns a FacetGrid object, not Axes")
print(f"catplot returns: {type(g)}")

---
## 9. Multi-Plot Grids: pairplot & FacetGrid

### pairplot
Creates a grid of **all pairwise scatter plots** — perfect for quickly exploring relationships in a dataset. On the diagonal, it shows the distribution of each variable.

**Analogy**: Like having a conversation with all your data columns at once — every column asks every other column "what's your relationship with me?"

In [ ]:
# ── pairplot: the EDA workhorse ────────────────────────────────────────────────
# Use a subset of tips for clarity
g = sns.pairplot(
    data=tips[['total_bill', 'tip', 'size', 'sex']],
    hue='sex',           # Color by gender
    diag_kind='kde',     # Diagonal: KDE (vs 'hist')
    plot_kws={'alpha': 0.5},
    palette='Set1',
    corner=True          # Only lower triangle (saves space)
)
g.fig.suptitle('Pairplot: All Pairwise Relationships in Tips Dataset', y=1.02)
plt.show()

# ── FacetGrid: custom multi-panel grids ────────────────────────────────────────
# Problem: show tip distribution for each day, separately for smokers/non-smokers
g2 = sns.FacetGrid(tips, col='day', row='smoker', 
                    height=3, aspect=1.2,
                    col_order=['Thur', 'Fri', 'Sat', 'Sun'])
g2.map(sns.histplot, 'tip', bins=12, kde=True)  # Apply the same plot to each panel
g2.add_legend()
g2.set_titles(col_template='{col_name}', row_template='Smoker: {row_name}')
g2.fig.suptitle('FacetGrid: Tip Distribution by Day and Smoking Status', y=1.02)
plt.show()

---
## 10. Themes & Color Palettes

### Themes (styles)
`sns.set_theme(style=...)` — options: `'white'`, `'whitegrid'`, `'dark'`, `'darkgrid'`, `'ticks'`

### Palettes
Color is communication. Choose palettes based on your data type:
- **Categorical** (no order): `'Set1'`, `'Set2'`, `'tab10'`, `'deep'`, `'colorblind'`
- **Sequential** (ordered, low→high): `'Blues'`, `'Reds'`, `'viridis'`, `'plasma'`
- **Diverging** (centered at 0): `'RdBu'`, `'PiYG'`, `'coolwarm'`

In [ ]:
# ── Show color palettes ────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 5))

categorical_palettes = ['deep', 'Set1', 'Set2', 'tab10', 'colorblind']
sequential_palettes  = ['Blues', 'Reds', 'viridis', 'plasma', 'YlOrRd']
diverging_palettes   = ['RdBu', 'PiYG', 'coolwarm', 'BrBG', 'RdYlGn']

for ax, palettes, label in zip(axes, 
    [categorical_palettes, sequential_palettes, diverging_palettes],
    ['Categorical', 'Sequential', 'Diverging']):
    n = 8
    offset = 0
    for name in palettes:
        colors = sns.color_palette(name, n)
        for i, color in enumerate(colors):
            ax.add_patch(plt.Rectangle((offset + i, 0), 1, 1, color=color))
        ax.text(offset + n/2, -0.2, name, ha='center', fontsize=8)
        offset += n + 1
    ax.set_xlim(0, offset)
    ax.set_ylim(-0.4, 1)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(label, fontweight='bold', fontsize=10)

plt.suptitle('Seaborn Color Palettes', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

# ── Apply different themes ─────────────────────────────────────────────────────
styles = ['white', 'whitegrid', 'dark', 'darkgrid', 'ticks']
fig, axes = plt.subplots(1, 5, figsize=(17, 3))

for ax, style in zip(axes, styles):
    with sns.axes_style(style):  # Temporarily apply style
        plt.sca(ax)  # Set current axes
        sns.lineplot(data=tips, x='total_bill', y='tip', ax=ax, alpha=0.7)
        ax.set_title(f'style="{style}"', fontsize=9)

plt.suptitle('Seaborn Style Themes', fontsize=11)
plt.tight_layout()
plt.show()

---
## 11. Combining Seaborn and Matplotlib

Because Seaborn returns Matplotlib objects, you can always customize further using Matplotlib.

**Pattern**:
1. Create the plot with Seaborn (fast, beautiful)
2. Get the `ax` object
3. Customize with `ax.set_*()`, `ax.annotate()`, etc.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Step 1: Seaborn plot
sns.violinplot(data=tips, x='day', y='total_bill', hue='time',
                split=True, palette='Set2', ax=ax,
                order=['Thur', 'Fri', 'Sat', 'Sun'])

# Step 2: Matplotlib customization
ax.set_title('Restaurant Bills by Day', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Total Bill ($)', fontsize=12)

# Add a reference line
median_bill = tips['total_bill'].median()
ax.axhline(median_bill, color='darkred', linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(3.5, median_bill + 0.5, f'Median: ${median_bill:.2f}', 
         ha='right', color='darkred', fontsize=10)

# Add annotation for an insight
ax.annotate('Weekend traffic\npushes bills higher',
             xy=(2, 48), xytext=(0.5, 52),
             arrowprops=dict(arrowstyle='->', color='navy'),
             fontsize=9, color='navy',
             bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='navy', alpha=0.8))

# Remove top and right spines
ax.spines[['top', 'right']].set_visible(False)

# Move legend inside
ax.legend(title='Meal Time', loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

---
## 12. Common Pitfalls & Best Practices

| Pitfall | Problem | Fix |
|---------|---------|-----|
| Using figure-level function with `ax=` | Doesn't work — catplot/relplot/displot ignore `ax=` | Use axes-level equivalent (histplot, scatterplot, etc.) |
| Forgetting `hue_order` | Categories appear in random order | `hue_order=['Male', 'Female']` to control order |
| `pairplot` on all columns | Slow with many columns; categorical columns error | Select numeric columns first |
| `heatmap` on non-numeric data | Error | Ensure the DataFrame passed is all numeric |
| Not centering heatmap diverging colormaps | Color scale is misleading | Set `center=0` for diverging, or `vmin/vmax` explicitly |
| Changing `sns.set_theme()` affects all subsequent plots | Unexpected style changes | Use `with sns.axes_style():` context manager for temporary changes |
| Interpreting CI bands as prediction intervals | CIs are for the mean, not individual observations | Prediction intervals are wider; use statsmodels for those |

---
## 13. Mini Project: Complete EDA Pipeline

**Scenario**: You're a data analyst at an online retailer. You receive a dataset of **customer transactions** and need to produce an EDA report with visualizations that tell the story of the data.

**Dataset**: 500 synthetic transactions with: customer age, purchase amount, product category, purchase day, discount applied, and whether the customer returned (churn).

**Goal**: Use Seaborn to systematically explore distributions, relationships, and patterns.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.0)
np.random.seed(42)

# ── Generate Dataset ──────────────────────────────────────────────────────────
n = 500
categories   = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
days_of_week = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

cat_choice = np.random.choice(categories, n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
base_prices = {'Electronics': 250, 'Clothing': 80, 'Home': 120, 'Sports': 90, 'Books': 30}

df = pd.DataFrame({
    'customer_age':  np.random.normal(35, 12, n).clip(18, 70).astype(int),
    'category':      cat_choice,
    'day':           np.random.choice(days_of_week, n, p=[0.12, 0.10, 0.10, 0.12, 0.18, 0.22, 0.16]),
    'discount_pct':  np.random.choice([0, 5, 10, 15, 20, 25], n, p=[0.3, 0.2, 0.2, 0.15, 0.1, 0.05]),
    'rating':        np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.20, 0.40, 0.25]),
})

# Purchase amount depends on category and discount
df['purchase_amount'] = [
    max(5, base_prices[cat] * (1 - disc/100) + np.random.normal(0, base_prices[cat]*0.2))
    for cat, disc in zip(df['category'], df['discount_pct'])
]

# Churn: older customers + low rating → more likely to churn
churn_prob = (df['customer_age'] - 18) / 100 + (5 - df['rating']) / 20
df['churned'] = np.random.binomial(1, churn_prob.clip(0.05, 0.6)).astype(bool)

# Age group column
df['age_group'] = pd.cut(df['customer_age'], bins=[18, 25, 35, 50, 70],
                          labels=['18-25', '26-35', '36-50', '51+'])

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print(f"\nChurn rate: {df['churned'].mean():.1%}")

# ── EDA Dashboard ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(17, 12))
gs  = plt.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# 1. Distribution of purchase amounts
ax1 = fig.add_subplot(gs[0, :2])
sns.histplot(data=df, x='purchase_amount', hue='category', ax=ax1,
              multiple='stack', bins=25, palette='Set2')
ax1.set_title('Purchase Amount Distribution by Category')
ax1.set_xlabel('Purchase Amount ($)')

# 2. Customer age distribution
ax2 = fig.add_subplot(gs[0, 2:])
sns.kdeplot(data=df, x='customer_age', hue='churned', ax=ax2, fill=True, alpha=0.5,
             palette={True: 'red', False: 'green'}, common_norm=False)
ax2.set_title('Age Distribution: Churned vs Retained')
ax2.set_xlabel('Customer Age')
ax2.legend(['Retained (False)', 'Churned (True)'])

# 3. Revenue by day of week
ax3 = fig.add_subplot(gs[1, :2])
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
sns.barplot(data=df, x='day', y='purchase_amount', order=day_order, ax=ax3,
             estimator='mean', errorbar='ci', palette='Blues_d')
ax3.set_title('Avg Purchase Amount by Day of Week')
ax3.set_xlabel('')
ax3.set_ylabel('Avg Purchase ($)')

# 4. Category × Discount heatmap
ax4 = fig.add_subplot(gs[1, 2:])
pivot = df.pivot_table(values='purchase_amount', index='category',
                         columns='discount_pct', aggfunc='mean')
sns.heatmap(pivot, ax=ax4, annot=True, fmt='.0f', cmap='YlOrRd',
             cbar_kws={'label': 'Avg Purchase ($)'})
ax4.set_title('Avg Purchase: Category × Discount Level')
ax4.set_xlabel('Discount %')

# 5. Churn rate by age group and category
ax5 = fig.add_subplot(gs[2, :2])
churn_by_group = df.groupby(['age_group', 'category'])['churned'].mean().reset_index()
churn_pivot = churn_by_group.pivot(index='age_group', columns='category', values='churned')
sns.heatmap(churn_pivot, ax=ax5, annot=True, fmt='.0%', cmap='RdYlGn_r',
             vmin=0, vmax=0.5, cbar_kws={'label': 'Churn Rate'})
ax5.set_title('Churn Rate: Age Group × Category')

# 6. Rating distribution by category
ax6 = fig.add_subplot(gs[2, 2:])
sns.countplot(data=df, x='rating', hue='category', ax=ax6, palette='Set2')
ax6.set_title('Rating Distribution by Category')
ax6.set_xlabel('Rating')
ax6.legend(title='Category', fontsize=8, bbox_to_anchor=(1, 1))

fig.suptitle('E-Commerce Transactions: EDA Dashboard', fontsize=14, fontweight='bold')
plt.savefig('/tmp/eda_dashboard.png', dpi=120, bbox_inches='tight')
print("\nEDA Dashboard saved to /tmp/eda_dashboard.png")
plt.show()

# ── Key Insights ──────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("KEY INSIGHTS FROM EDA:")
print("="*50)
print(f"1. Electronics = highest spend (avg ${df[df.category=='Electronics']['purchase_amount'].mean():.0f})")
print(f"2. Weekend sales are {(df[df['day'].isin(['Sat','Sun'])]['purchase_amount'].mean() / df['purchase_amount'].mean() - 1):.0%} above average")
print(f"3. Customers 51+ have highest churn: {df[df.age_group=='51+']['churned'].mean():.0%}")
print(f"4. Discounts > 15% strongly increase Electronics sales")

---
## 14. Interview Q&A

**Q1: What is the difference between Seaborn and Matplotlib?**  
**A**: Matplotlib is the low-level plotting library that gives you full control. Seaborn is a high-level wrapper around Matplotlib that provides beautiful defaults, automatic statistical operations (like confidence intervals), and easy grouping via `hue`. They're complementary: Seaborn makes 80% of plots fast and beautiful; Matplotlib handles the remaining 20% that need custom layouts.

---
**Q2: What is the difference between `axes-level` and `figure-level` functions in Seaborn?**  
**A**: Axes-level functions (like `histplot`, `boxplot`, `scatterplot`) take an `ax=` argument and draw into an existing Matplotlib Axes. Figure-level functions (`displot`, `catplot`, `relplot`) create their own Figure and can automatically create multi-panel grids using `col=` and `row=`. Figure-level functions return a `FacetGrid` object, not an `Axes`.

---
**Q3: What does `hue` do in Seaborn?**  
**A**: `hue` adds a third dimension to a plot by color-coding data points based on a categorical variable. For example, `sns.scatterplot(x='bill', y='tip', hue='day')` colors points differently for each day. Seaborn automatically creates a legend and chooses distinct colors.

---
**Q4: When would you use a violin plot vs a box plot?**  
**A**: Box plots show summary statistics (median, quartiles, outliers) but hide the shape. Violin plots show the full distribution shape via KDE. Use violin plots when the distribution shape matters (e.g., bimodal distributions) or when you want to show data density. Box plots are better when you want clear outlier visibility or when comparing many groups.

---
**Q5: How does Seaborn calculate the confidence interval in `lineplot` and `barplot`?**  
**A**: By default, Seaborn uses **bootstrapping** to compute 95% confidence intervals. It repeatedly resamples the data (with replacement) many times and computes the statistic each time, then shows the range that contains 95% of those estimates. This is controlled by `errorbar='ci'` (bootstrap) vs `errorbar='se'` (standard error) vs `errorbar='sd'` (standard deviation).

---
**Q6: What is a pairplot and when would you use it?**  
**A**: A pairplot creates a grid of scatter plots for every pair of numeric variables. The diagonal shows each variable's distribution. It's used at the start of EDA to quickly spot correlations, outliers, and group separations. Add `hue` to see if groups (e.g., classes) are linearly separable — this is essentially a quick PCA-free view of the data.

---
**Q7: How do you use Seaborn to check assumptions for a linear regression?**  
**A**: Use `residplot(x, y)` which plots the residuals (actual - predicted). If the linear model is appropriate, residuals should be randomly scattered around zero (no pattern). If there's a U-shape or systematic pattern, you need polynomial regression or a transformation.

---
## 15. Resources

### Official
- **Documentation**: https://seaborn.pydata.org/
- **API Reference**: https://seaborn.pydata.org/api.html
- **Tutorial** (by the author): https://seaborn.pydata.org/tutorial.html
- **Gallery** (200+ examples): https://seaborn.pydata.org/examples/

### YouTube
- **Keith Galli — Seaborn Tutorial**: https://www.youtube.com/watch?v=6GUZXDef2U0
- **Data Professor — Seaborn EDA**: https://www.youtube.com/watch?v=Bx_1bKoRsNI
- **Patrick Loeber — Seaborn Crash Course**: https://www.youtube.com/watch?v=ooqXQ37XHMM

### Articles
- **Python Data Science Handbook — Seaborn chapter**: https://jakevdp.github.io/PythonDataScienceHandbook/04.14-visualization-with-seaborn.html
- **Seaborn Paper**: Waskom, M. L. (2021). Seaborn: Statistical data visualization. *Journal of Open Source Software*, 6(60), 3021. https://doi.org/10.21105/joss.03021

---
## Summary & What's Next

### What You Learned
| Concept | Key Point |
|---------|----------|
| Mental model | `data=df, x=col, y=col, hue=cat` — works on any Seaborn function |
| Distribution plots | `histplot`, `kdeplot`, `ecdfplot` for understanding one variable |
| Categorical plots | `boxplot`, `violinplot`, `barplot` for comparing groups |
| Relational plots | `scatterplot`, `lineplot` with automatic CIs |
| Regression | `regplot`, `lmplot`, `residplot` |
| Matrix plots | `heatmap` for correlation and pivot tables |
| Figure-level | `catplot`, `displot`, `relplot` for automatic multi-panel grids |
| Grids | `pairplot` for all-pairwise, `FacetGrid` for custom grids |
| Themes | `sns.set_theme()`, palettes for categorical/sequential/diverging data |
| Seaborn + MPL | Always get the `ax` back; customize freely with Matplotlib |

### What's Next?
- **Next Notebook**: Plotly — interactive, web-ready visualizations
- **Practice**: Load the `flights` or `penguins` dataset from Seaborn and explore it with at least 5 different plot types
- **Challenge**: Build a complete EDA notebook for a Kaggle dataset using only Seaborn + Matplotlib

> **Key insight**: Seaborn is the data scientist's go-to for EDA. Once you know its API, you can explore any dataset in minutes. The `hue`, `col`, and `row` parameters alone replace hours of manual plotting code.